# GeoLab Tutorial: Automated Acrostichum Delineation from Drone Orthomosaics

This notebook automates the detection and delineation of *Acrostichum* fern (an indicator of mangrove degradation) and mangrove canopy directly from a large drone orthomosaic GeoTIFF, using an AI object-detection model from **Roboflow**.

**Workflow:**
1. Clip the orthomosaic to a study-area boundary (Shapefile)
2. Tile the clipped region for efficient, memory-safe AI inference
3. Run Roboflow instance-segmentation on each tile
4. Merge overlapping detections with DBSCAN clustering
5. Clip *Acrostichum* polygons to the study boundary
6. Place random 5×10 m sample plots within *Acrostichum* areas using an R-tree index
7. Save Shapefiles + GeoJSON and produce a final overview map

**Output:** `Acrostichum_Delineation-Clipped.shp`, `Mangrove_Delineation.shp`, `Random_Polygons_Within_Acrostichum.shp`, and `delineation_overview.png`

> **Target audience:** GIS/QGIS users learning to combine drone imagery, AI object detection, and geospatial Python for vegetation mapping.

## Requirements

```bash
pip install rasterio geopandas shapely roboflow opencv-python scikit-learn rtree tqdm matplotlib pandas
```

| Package | Purpose |
|---|---|
| `rasterio` | Windowed (memory-safe) reads from large GeoTIFFs |
| `geopandas` | Loading/saving Shapefiles, spatial joins, clipping |
| `shapely` | Geometry creation, merging, affine transforms |
| `roboflow` | Connecting to the trained Acrostichum detection model |
| `opencv-python` | Writing tile JPEG files and resizing images |
| `scikit-learn` | DBSCAN clustering to merge nearby detections |
| `rtree` | Spatial index for fast non-overlap checks during plot placement |
| `tqdm` | Progress bars for tile processing loops |
| `matplotlib` | Final overview visualization |

## Step 1: Import Libraries

In [ ]:
import os
import math
import numpy as np
import cv2                           # Write tile JPEGs and resize images for display

# Geospatial
import rtree                         # R-tree spatial index — enables O(log n) overlap checks
import geopandas as gpd
import rasterio
from rasterio.windows import Window  # Read only a sub-region of a large raster (avoids loading the whole file)
from shapely.geometry import box, Polygon, MultiPolygon
from shapely.ops import unary_union  # Dissolve multiple polygons into one
from shapely import affinity          # Apply affine transforms to Shapely geometries

# ML / data
import pandas as pd
from sklearn.cluster import DBSCAN   # Density-based clustering to merge nearby detections
from roboflow import Roboflow        # API client for the trained object-detection model

from tqdm import tqdm                # Progress bar — essential for long tiling/inference loops
import warnings
import matplotlib.pyplot as plt

# Suppress a minor GeoDataFrame warning that doesn't affect results
warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)

print("✓ All libraries imported successfully")

## Step 2: Define the VegetationDelineator Class

All processing logic is encapsulated in a single class so the workflow can be triggered with a single `run_full_delineation()` call. The class is intentionally designed to be **memory-safe**: it never loads the entire orthomosaic into RAM — it reads only the pixels needed for each tile using Rasterio's `Window` API.

### Key design decisions
| Decision | Reason |
|---|---|
| Windowed reads | Drone orthomosaics can be >10 GB — loading the whole image causes `MemoryError` |
| DBSCAN clustering | Merges duplicate detections from overlapping tiles into a single polygon |
| R-tree index | Checking `n` placed plots against each other naively is O(n²); R-tree makes it O(n log n) |
| Tile JPEG temp files | Roboflow's SDK accepts file paths, not numpy arrays, so tiles must be saved temporarily |

In [ ]:
class VegetationDelineator:
    """
    End-to-end pipeline for detecting and delineating Acrostichum fern and
    mangrove canopy from large drone orthomosaics using AI-based instance
    segmentation (Roboflow) and geospatial Python.

    Workflow (called via run_full_delineation):
        1. Read the raster's georeference (CRS, transform, dimensions)
        2. Convert the study-area Shapefile extent to pixel coordinates
        3. Tile the clipped region using Rasterio windowed reads
        4. Run Roboflow inference on each tile (returns bbox + polygon points)
        5. Convert tile-local detections to global pixel coordinates
        6. Cluster nearby detections with DBSCAN and merge into final polygons
        7. Re-project and clip Acrostichum polygons to the study boundary
        8. Generate random 5×10 m sample plots within Acrostichum areas
        9. Save Shapefiles, GeoJSONs, and a final visualization PNG
    """

    def __init__(self, api_key: str, project_id: str, version: int):
        """
        Initialize and download the Roboflow model.

        Parameters
        ----------
        api_key    : str — Your Roboflow API key (from account settings)
        project_id : str — Roboflow project ID (e.g. 'acrostichum-2-g8ewp')
        version    : int — Model version number to use
        """
        print("Initializing Roboflow model...")
        rf = Roboflow(api_key=api_key)
        project = rf.workspace().project(project_id)
        self.model = project.version(version).model
        print("✓ Roboflow model ready")

        # Tile size in pixels. 1024×1024 balances inference speed and spatial coverage.
        # Larger tiles may exceed Roboflow's API image size limits.
        self.tile_size = (1024, 1024)
        self.overlap = 0  # No overlap needed — DBSCAN handles cross-tile merging

    # ------------------------------------------------------------------
    # Step A: Read raster metadata
    # ------------------------------------------------------------------

    def get_image_georeference(self, image_path: str) -> dict:
        """
        Read spatial metadata from a GeoTIFF without loading pixel data.

        The georeference dict is passed around the pipeline to convert between
        pixel coordinates and geographic (map) coordinates.

        Returns dict with keys: transform, crs, width, height, bounds.
        Returns None if the file cannot be opened.
        """
        print(f"Reading georeference from: {image_path}")
        try:
            with rasterio.open(image_path) as src:
                print(f"  CRS       : {src.crs}")
                print(f"  Dimensions: {src.width} cols × {src.height} rows")
                return {
                    'transform': src.transform,
                    'crs'      : src.crs,
                    'width'    : src.width,
                    'height'   : src.height,
                    'bounds'   : src.bounds
                }
        except Exception as e:
            print(f"❌ Cannot open raster: {e}")
            return None

    # ------------------------------------------------------------------
    # Step B: Convert Shapefile extent → pixel ROI
    # ------------------------------------------------------------------

    def get_shapefile_bounds(self, shapefile_path: str, image_path: str) -> tuple:
        """
        Convert the geographic extent of a Shapefile to pixel (column/row) coordinates
        within the raster.

        This lets us tile only the relevant sub-region of a large image, rather than
        tiling the entire raster including areas outside the study boundary.

        Returns (min_col, min_row, max_col, max_row) clamped to image dimensions.
        """
        print(f"Loading study-area shapefile: {shapefile_path}")
        study_gdf = gpd.read_file(shapefile_path)

        with rasterio.open(image_path) as src:
            # Reproject the shapefile to match the raster's CRS if they differ
            if study_gdf.crs != src.crs:
                print(f"  Reprojecting shapefile from {study_gdf.crs} → {src.crs}")
                study_gdf = study_gdf.to_crs(src.crs)

            geo_bounds = study_gdf.total_bounds  # [xmin, ymin, xmax, ymax]

            # rasterio.DatasetReader.index() converts geographic coords to row/col.
            # Note: top-left corner is (0, 0); rows increase downward.
            min_col, min_row = src.index(geo_bounds[0], geo_bounds[3])  # top-left
            max_col, max_row = src.index(geo_bounds[2], geo_bounds[1])  # bottom-right

            # Clamp to valid image dimensions to avoid out-of-bounds window reads
            min_col = max(0, min_col)
            min_row = max(0, min_row)
            max_col = min(src.width,  max_col)
            max_row = min(src.height, max_row)

            print(f"  Pixel ROI: cols {min_col}–{max_col}, rows {min_row}–{max_row}")
            return (min_col, min_row, max_col, max_row)

    # ------------------------------------------------------------------
    # Step C: Tile the ROI
    # ------------------------------------------------------------------

    def create_tiles_from_bounds(self, image_path: str, pixel_bounds: tuple) -> list:
        """
        Divide the region of interest into JPEG tiles for Roboflow inference.

        Uses Rasterio windowed reads (Window API) so only one tile's worth of
        pixels is in RAM at a time — critical for orthomosaics that are larger
        than available RAM.

        Tiles smaller than 1/4 of tile_size in either dimension are skipped to
        avoid sending near-empty images to the model.

        Returns list of tile dicts: {id, filename, bounds (pixel), size (pixel)}.
        """
        print("\n✂️  Tiling the region of interest...")
        try:
            with rasterio.open(image_path) as src:
                minx, miny, maxx, maxy = pixel_bounds
                tile_w, tile_h = self.tile_size
                x_tiles = math.ceil((maxx - minx) / tile_w)
                y_tiles = math.ceil((maxy - miny) / tile_h)
                print(f"  Grid: {x_tiles} × {y_tiles} = {x_tiles * y_tiles} tiles")

                tiles = []
                tile_id = 0
                for y in range(y_tiles):
                    for x in range(x_tiles):
                        start_x = minx + x * tile_w
                        start_y = miny + y * tile_h
                        end_x = min(start_x + tile_w, maxx)
                        end_y = min(start_y + tile_h, maxy)

                        # Skip slivers — too small to contain meaningful detections
                        if (end_x - start_x) < tile_w // 4 or (end_y - start_y) < tile_h // 4:
                            continue

                        # Read only this tile's pixels using a Rasterio Window
                        window = Window(start_x, start_y, end_x - start_x, end_y - start_y)
                        # Read RGB bands; shape is (3, H, W) — bands first
                        tile_rgb = src.read([1, 2, 3], window=window)
                        # Rearrange to (H, W, 3) for OpenCV compatibility
                        tile_rgb = np.moveaxis(tile_rgb, 0, -1)
                        # OpenCV uses BGR channel order; rasterio returns RGB — convert
                        tile_bgr = cv2.cvtColor(tile_rgb, cv2.COLOR_RGB2BGR)

                        tile_filename = f"tile_{tile_id:04d}.jpg"
                        # JPEG quality 90: good balance of file size vs. detail for inference
                        cv2.imwrite(tile_filename, tile_bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])

                        tiles.append({
                            'id'      : tile_id,
                            'filename': tile_filename,
                            'bounds'  : (start_x, start_y, end_x, end_y),
                            'size'    : (end_x - start_x, end_y - start_y)
                        })
                        tile_id += 1

            print(f"  Generated {len(tiles)} tiles")
            return tiles

        except Exception as e:
            print(f"❌ Tiling failed: {e}")
            return []

    # ------------------------------------------------------------------
    # Step D: Run inference on one tile
    # ------------------------------------------------------------------

    def process_tile(self, tile_info: dict, confidence: float = 0.5) -> list:
        """
        Send a single tile image to Roboflow and convert predictions to
        global pixel coordinates.

        Roboflow returns bounding boxes and (optionally) segmentation polygon
        points in tile-local coordinates (origin at tile top-left). We shift
        these by the tile's global pixel offset to get image-wide coordinates.

        Parameters
        ----------
        tile_info  : dict — tile metadata from create_tiles_from_bounds()
        confidence : float — minimum detection confidence to include (0–1)

        Returns list of detection dicts with global_bbox and optional global_polygon.
        """
        try:
            prediction = self.model.predict(tile_info['filename'], confidence=confidence)
            detections = []
            # tile origin in global pixel space
            tile_origin_x, tile_origin_y = tile_info['bounds'][:2]

            for pred in prediction.json()['predictions']:
                detection = {
                    'class'      : pred['class'],
                    'confidence' : pred['confidence'],
                    # Shift bbox center from tile-local to global pixel coordinates
                    'global_bbox': {
                        'x'     : pred['x'] + tile_origin_x,
                        'y'     : pred['y'] + tile_origin_y,
                        'width' : pred['width'],
                        'height': pred['height']
                    },
                    'tile_id': tile_info['id']
                }
                # If the model returns polygon points (instance segmentation), shift those too
                if 'points' in pred:
                    detection['global_polygon'] = [
                        (p['x'] + tile_origin_x, p['y'] + tile_origin_y)
                        for p in pred['points']
                    ]
                detections.append(detection)
            return detections

        except Exception as e:
            print(f"  Warning: inference failed on {tile_info['filename']}: {e}")
            return []

    # ------------------------------------------------------------------
    # Step E: Merge detections into polygons
    # ------------------------------------------------------------------

    def polygonize_detections(self, detections: list, merge_distance: int = 50) -> list:
        """
        Convert raw detections to merged Shapely polygons using DBSCAN clustering.

        Why DBSCAN? The same vegetation patch may produce multiple overlapping
        detections from neighbouring tiles or across a single tile. DBSCAN groups
        nearby detection centres, then we dissolve each cluster into one polygon
        using unary_union.

        Parameters
        ----------
        detections     : list — output of process_tile() calls
        merge_distance : int  — DBSCAN eps in pixels; detections closer than this
                                are merged. Tune based on expected vegetation patch size.

        Returns list of feature dicts: {class, confidence, geometry, detection_count, source_detections}.
        """
        if not detections:
            return []
        print(f"\n🔗 Merging {len(detections)} detections (DBSCAN eps={merge_distance}px)...")

        # Split detections by class so Acrostichum and Mangrove don't merge together
        class_groups = {}
        for d in detections:
            class_groups.setdefault(d['class'], []).append(d)

        merged_features = []
        for class_name, class_dets in class_groups.items():
            # Use bounding box centres as the clustering coordinates
            centres = np.array([[d['global_bbox']['x'], d['global_bbox']['y']] for d in class_dets])

            # min_samples=1 ensures every point is in some cluster (no noise points)
            labels = DBSCAN(eps=merge_distance, min_samples=1).fit(centres).labels_

            for cluster_id in set(labels):
                cluster_dets = [d for i, d in enumerate(class_dets) if labels[i] == cluster_id]

                # Build a Shapely polygon for each detection in the cluster
                polygons = []
                for det in cluster_dets:
                    if 'global_polygon' in det and len(det['global_polygon']) >= 3:
                        # Prefer the segmentation polygon — more accurate than bbox
                        polygons.append(Polygon(det['global_polygon']))
                    else:
                        # Fall back to bounding box if segmentation is unavailable
                        b = det['global_bbox']
                        polygons.append(box(
                            b['x'] - b['width']  / 2, b['y'] - b['height'] / 2,
                            b['x'] + b['width']  / 2, b['y'] + b['height'] / 2
                        ))

                if not polygons:
                    continue

                # Dissolve all polygons in this cluster into one geometry.
                # .buffer(0) fixes any self-intersection artefacts.
                merged_poly = unary_union(polygons).buffer(0)
                avg_conf = float(np.mean([d['confidence'] for d in cluster_dets]))

                merged_features.append({
                    'class'            : class_name,
                    'confidence'       : avg_conf,
                    'geometry'         : merged_poly,
                    'detection_count'  : len(cluster_dets),
                    'source_detections': [d['tile_id'] for d in cluster_dets]
                })

        print(f"  Produced {len(merged_features)} merged polygons")
        return merged_features

    # ------------------------------------------------------------------
    # Step F: Pixel → geographic coordinate conversion
    # ------------------------------------------------------------------

    def geometry_to_geographic(self, pixel_geometry, georeference: dict):
        """
        Convert a Shapely geometry from pixel coordinates to geographic (map) coordinates.

        Uses the raster's affine GeoTransform, which maps (col, row) pixel positions
        to (easting, northing) map coordinates in the raster's CRS.

        Handles both Polygon and MultiPolygon geometry types.
        """
        if not georeference or not georeference['transform']:
            return pixel_geometry

        tf = georeference['transform']

        def _transform_coords(coords):
            """Apply the GDAL affine transform to a list of (col, row) tuples."""
            return [tf * c for c in coords]

        if pixel_geometry.geom_type == 'Polygon':
            return Polygon(
                _transform_coords(pixel_geometry.exterior.coords),
                [_transform_coords(ring.coords) for ring in pixel_geometry.interiors]
            )
        elif pixel_geometry.geom_type == 'MultiPolygon':
            return MultiPolygon([
                Polygon(
                    _transform_coords(p.exterior.coords),
                    [_transform_coords(r.coords) for r in p.interiors]
                )
                for p in pixel_geometry.geoms
            ])
        return pixel_geometry

    # ------------------------------------------------------------------
    # Step G: Clip Acrostichum polygons to study boundary
    # ------------------------------------------------------------------

    def clip_acrostichum_detections(self, merged_features: list, shapefile_path: str, georeference: dict):
        """
        Clip detected Acrostichum polygons to the study-area boundary Shapefile.

        This removes any artefact detections that fall partially or fully outside
        the legitimate rehabilitation area.

        Returns (clipped_acrostichum_gdf, mangrove_gdf, study_boundary_gdf).
        """
        print(f"\n✂️  Clipping Acrostichum detections to boundary: {shapefile_path}")
        boundary_gdf = gpd.read_file(shapefile_path)

        # Convert pixel geometries → geographic coordinates
        geo_geometries = [
            self.geometry_to_geographic(f['geometry'], georeference)
            for f in merged_features
        ]
        attributes = [
            {'class': f['class'], 'confidence': f['confidence'], 'det_count': f['detection_count']}
            for f in merged_features
        ]
        detections_gdf = gpd.GeoDataFrame(attributes, geometry=geo_geometries, crs=georeference['crs'])

        # Align CRS before clipping to avoid misaligned geometries
        if detections_gdf.crs != boundary_gdf.crs:
            detections_gdf = detections_gdf.to_crs(boundary_gdf.crs)

        acrostichum_gdf = detections_gdf[detections_gdf['class'] == 'Acrostichum'].copy()
        mangrove_gdf    = detections_gdf[detections_gdf['class'] == 'Mangrove'].copy()
        print(f"  Before clip: {len(acrostichum_gdf)} Acrostichum, {len(mangrove_gdf)} Mangrove polygons")

        if not acrostichum_gdf.empty:
            clipped_acrostichum = gpd.clip(acrostichum_gdf, boundary_gdf)
            print(f"  After clip : {len(clipped_acrostichum)} Acrostichum polygons")
        else:
            clipped_acrostichum = acrostichum_gdf
            print("  No Acrostichum polygons found.")

        return clipped_acrostichum, mangrove_gdf, boundary_gdf

    # ------------------------------------------------------------------
    # Step H: Create a 100 m reference grid
    # ------------------------------------------------------------------

    def create_qgis_style_grid(self, bounds, cell_width_m: float, cell_height_m: float, crs) -> gpd.GeoDataFrame:
        """
        Create a regular geographic grid of specified cell size (in metres) over a bounding box.

        The conversion from metres to degrees uses the cosine correction for longitude
        (1 degree of longitude is shorter near the poles), while latitude degrees
        are treated as constant at 111,320 m.

        This grid is used as a stratified sampling framework: sample plots are placed
        randomly within each 100 m cell that overlaps with Acrostichum areas.
        """
        print(f"  Building {cell_width_m}×{cell_height_m} m reference grid...")
        minx, miny, maxx, maxy = bounds

        # Approximate degree sizes at the study area centre latitude
        lat_center    = (miny + maxy) / 2
        m_per_deg_lon = 111320 * np.cos(np.radians(lat_center))
        m_per_deg_lat = 111320.0

        cell_w_deg = cell_width_m  / m_per_deg_lon
        cell_h_deg = cell_height_m / m_per_deg_lat

        nx = int(math.ceil((maxx - minx) / cell_w_deg))
        ny = int(math.ceil((maxy - miny) / cell_h_deg))
        print(f"  Grid dimensions: {nx} × {ny} cells")

        grid_cells = [
            box(minx + i * cell_w_deg, miny + j * cell_h_deg,
                minx + (i + 1) * cell_w_deg, miny + (j + 1) * cell_h_deg)
            for i in range(nx) for j in range(ny)
        ]
        return gpd.GeoDataFrame(geometry=grid_cells, crs=crs)

    # ------------------------------------------------------------------
    # Step I: Place random sample plots (R-tree index)
    # ------------------------------------------------------------------

    def place_polygons_in_clipped_grids(
        self,
        clipped_grids_gdf: gpd.GeoDataFrame,
        placement_boundary_gdf: gpd.GeoDataFrame,
        cell_size_m: tuple,
        spacing_m: float,
        target_total_polygons: int
    ) -> gpd.GeoDataFrame:
        """
        Randomly place non-overlapping rectangular sample plots (e.g. 5×10 m) within
        the Acrostichum delineation area, using an R-tree spatial index for efficiency.

        Why R-tree? Checking every new candidate plot against all previously placed
        plots is O(n²). An R-tree reduces this to O(n log n) by quickly filtering
        candidates to only those whose bounding boxes overlap.

        Algorithm:
            1. Pick a random 100 m grid cell (stratified random sampling)
            2. Generate a random point within that cell
            3. Create a 5×10 m plot at a random orientation
            4. Check the plot is fully within the Acrostichum area
            5. Check R-tree for any existing plot within spacing_m
            6. If clear: add to R-tree and final list; otherwise discard and retry

        Parameters
        ----------
        clipped_grids_gdf       : GeoDataFrame of 100 m grid cells intersecting Acrostichum
        placement_boundary_gdf  : GeoDataFrame of the Acrostichum area (containment check)
        cell_size_m             : (width_m, height_m) of each sample plot
        spacing_m               : minimum gap between plots (metres)
        target_total_polygons   : how many plots to place

        Returns GeoDataFrame of placed sample plots, or None if skipped.
        """
        print(f"\n🎲 Placing {target_total_polygons} sample plots ({cell_size_m[0]}×{cell_size_m[1]} m)...")

        if clipped_grids_gdf.empty or placement_boundary_gdf.empty:
            print("  ⚠️  Empty grid or boundary — skipping plot placement.")
            return None

        # R-tree index stores bounding boxes of already-placed (buffered) plots
        spatial_index = rtree.index.Index()
        placed_plots  = []
        crs = placement_boundary_gdf.crs

        # Convert m → degrees (approximate, at study area latitude)
        minx, miny, _, _ = placement_boundary_gdf.total_bounds
        m_per_deg_lon = 111320 * np.cos(np.radians(miny))
        m_per_deg_lat = 111320.0
        plot_w_deg  = cell_size_m[0] / m_per_deg_lon
        plot_h_deg  = cell_size_m[1] / m_per_deg_lat
        spacing_deg = spacing_m / m_per_deg_lon

        max_attempts = target_total_polygons * 100  # Prevent infinite loops in low-density areas
        attempts = 0
        pbar = tqdm(total=target_total_polygons, desc="Placing sample plots")

        while len(placed_plots) < target_total_polygons and attempts < max_attempts:
            attempts += 1

            # Stratified: pick a random 100 m cell, then a random point within it
            grid_cell = clipped_grids_gdf.sample(1).iloc[0].geometry
            gx1, gy1, gx2, gy2 = grid_cell.bounds
            cx = np.random.uniform(gx1, gx2)
            cy = np.random.uniform(gy1, gy2)

            # Create a rotated sample plot centred at (cx, cy)
            angle    = np.random.uniform(0, 360)
            base     = box(-plot_w_deg / 2, -plot_h_deg / 2, plot_w_deg / 2, plot_h_deg / 2)
            new_plot = affinity.translate(affinity.rotate(base, angle), xoff=cx, yoff=cy)

            # Discard if the plot extends outside the Acrostichum area
            if not placement_boundary_gdf.contains(new_plot).any():
                continue

            # Check for nearby plots using R-tree — O(log n) lookup
            buffered = new_plot.buffer(spacing_deg)
            if not list(spatial_index.intersection(buffered.bounds)):
                # No conflict: add buffered bounds to index, unbuffered plot to output
                spatial_index.insert(len(placed_plots), buffered.bounds)
                placed_plots.append(new_plot)
                pbar.update(1)

        pbar.close()

        if len(placed_plots) < target_total_polygons:
            print(f"  ⚠️  Placed {len(placed_plots)} of {target_total_polygons} plots (area may be too small).")
        else:
            print(f"  ✓ Placed {len(placed_plots)} sample plots")

        return gpd.GeoDataFrame(geometry=placed_plots, crs=crs)

    # ------------------------------------------------------------------
    # Step J: Save outputs
    # ------------------------------------------------------------------

    def save_to_shapefile(self, gdf: gpd.GeoDataFrame, output_path: str) -> None:
        """
        Save a GeoDataFrame to both Shapefile (.shp) and GeoJSON format.

        GeoJSON is saved alongside the Shapefile for easy loading in web tools
        (e.g., geojson.io) without needing a full GIS installation.
        """
        if gdf is None or gdf.empty:
            print(f"  No data to save: {output_path}")
            return
        try:
            gdf.to_file(output_path)
            print(f"💾 Shapefile saved : {output_path}")
            geojson_path = output_path.replace('.shp', '.geojson')
            gdf.to_file(geojson_path, driver='GeoJSON')
            print(f"💾 GeoJSON saved   : {geojson_path}")
        except Exception as e:
            print(f"❌ Save failed for {output_path}: {e}")

    # ------------------------------------------------------------------
    # Step K: Final visualization
    # ------------------------------------------------------------------

    def create_overview_visualization_with_random(
        self,
        image_path: str,
        clipped_acrostichum: gpd.GeoDataFrame,
        mangrove_gdf: gpd.GeoDataFrame,
        random_polygons_gdf: gpd.GeoDataFrame,
        georeference: dict,
        pixel_bounds: tuple
    ) -> None:
        """
        Create and save an overview PNG showing all delineation results overlaid
        on the orthomosaic background.

        The background image is read using a Rasterio Window (memory-safe).
        For display, very large images (>4000 px) are downsampled while
        maintaining aspect ratio.

        All GeoDataFrames are converted from geographic to display pixel
        coordinates using the inverse affine transform before plotting.
        """
        print("\n🎨 Generating overview visualization...")
        minx_pix, miny_pix, maxx_pix, maxy_pix = pixel_bounds

        # Read only the ROI region — safe for large files
        try:
            with rasterio.open(image_path) as src:
                window = Window(minx_pix, miny_pix, maxx_pix - minx_pix, maxy_pix - miny_pix)
                region_rgb = np.moveaxis(src.read([1, 2, 3], window=window), 0, -1)
        except Exception as e:
            print(f"  Warning: cannot read image region ({e}) — using blank background")
            region_rgb = np.zeros((maxy_pix - miny_pix, maxx_pix - minx_pix, 3), dtype=np.uint8)

        # Downsample very large display images to avoid matplotlib memory issues
        h, w = region_rgb.shape[:2]
        scale = min(4000 / w, 4000 / h, 1.0)  # never upscale
        if scale < 1.0:
            nw, nh = int(w * scale), int(h * scale)
            print(f"  Resizing display: {w}×{h} → {nw}×{nh}")
            display_img = cv2.resize(region_rgb, (nw, nh), interpolation=cv2.INTER_AREA)
        else:
            display_img = region_rgb

        def geo_to_pixel_plot(input_gdf):
            """Convert geographic coords → scaled display pixel coords for matplotlib."""
            if input_gdf is None or input_gdf.empty:
                return gpd.GeoDataFrame(geometry=[])
            gdf_local = input_gdf.to_crs(georeference['crs']) if input_gdf.crs != georeference['crs'] else input_gdf
            inv_tf = ~georeference['transform']  # invert the affine transform: geo → pixel
            # Step 1: geographic → global pixel
            pixel_geoms = gdf_local.geometry.apply(
                lambda g: affinity.affine_transform(g, [
                    inv_tf.a, inv_tf.b, inv_tf.d, inv_tf.e, inv_tf.xoff, inv_tf.yoff
                ])
            )
            # Step 2: global pixel → local display pixel (subtract ROI offset and scale)
            pixel_geoms = pixel_geoms.apply(
                lambda g: affinity.affine_transform(g, [
                    scale, 0, 0, scale, -minx_pix * scale, -miny_pix * scale
                ])
            )
            return gpd.GeoDataFrame(geometry=pixel_geoms)

        fig, ax = plt.subplots(figsize=(20, 20))
        ax.imshow(display_img)

        # Overlay each layer with a distinct colour
        geo_to_pixel_plot(clipped_acrostichum).plot(
            ax=ax, facecolor='red',  edgecolor='red',    alpha=0.5, label='Acrostichum')
        geo_to_pixel_plot(mangrove_gdf).plot(
            ax=ax, facecolor='blue', edgecolor='blue',   alpha=0.5, label='Mangrove')
        geo_to_pixel_plot(random_polygons_gdf).plot(
            ax=ax, facecolor='none', edgecolor='yellow', linewidth=1, label='Sample plots (5×10 m)')

        ax.set_title("Vegetation Delineation: Acrostichum (red), Mangrove (blue), Sample Plots (yellow)")
        ax.set_axis_off()
        plt.legend(loc='lower right')
        plt.tight_layout()

        out_png = "delineation_overview.png"
        plt.savefig(out_png, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Visualization saved: {out_png}")

    # ------------------------------------------------------------------
    # Main entry point
    # ------------------------------------------------------------------

    def run_full_delineation(
        self,
        image_path: str,
        shapefile_path: str,
        confidence: float = 0.5,
        merge_distance: int = 50,
        create_random_grid: bool = True,
        polygon_cell_size: tuple = (5, 10)
    ) -> None:
        """
        Run the complete Acrostichum delineation pipeline from start to finish.

        Parameters
        ----------
        image_path         : str   — Path to the input GeoTIFF orthomosaic
        shapefile_path     : str   — Path to the study-area boundary Shapefile
        confidence         : float — Roboflow detection confidence threshold (0–1)
        merge_distance     : int   — DBSCAN merge radius in pixels
        create_random_grid : bool  — Whether to generate random sample plots
        polygon_cell_size  : tuple — (width_m, height_m) of each sample plot
        """
        print("🌿 Starting Vegetation Delineation Pipeline")
        print("=" * 70)

        georeference = self.get_image_georeference(image_path)
        if not georeference:
            print("❌ Aborting: could not read georeference.")
            return

        pixel_bounds = self.get_shapefile_bounds(shapefile_path, image_path)

        tiles = self.create_tiles_from_bounds(image_path, pixel_bounds)
        if not tiles:
            print("❌ Aborting: no tiles created.")
            return

        # Run inference on every tile; collect all detections
        all_detections = []
        for tile in tqdm(tiles, desc="Running inference"):
            all_detections.extend(self.process_tile(tile, confidence=confidence))
        print(f"\n📊 Total raw detections: {len(all_detections)}")

        merged_features = self.polygonize_detections(all_detections, merge_distance=merge_distance)
        clipped_acrostichum, mangrove_gdf, boundary_gdf = self.clip_acrostichum_detections(
            merged_features, shapefile_path, georeference
        )

        # Save vegetation delineation outputs
        self.save_to_shapefile(clipped_acrostichum, "Acrostichum_Delineation-Clipped.shp")
        self.save_to_shapefile(mangrove_gdf,        "Mangrove_Delineation.shp")

        # --- Generate random sample plots (proportional to LUAS_RK area) ---
        random_polygons_gdf = None
        if create_random_grid and not clipped_acrostichum.empty:
            print("\n🌱 Generating random sample plots...")
            try:
                # Spatial join: find which original boundary polygons intersect detected areas.
                # drop_duplicates prevents double-counting if one polygon matches many detections.
                intersecting = gpd.sjoin(
                    boundary_gdf, clipped_acrostichum, how='inner', op='intersects'
                ).drop_duplicates(subset=['index_left'])

                # Target = sum of LUAS_RK (rehabilitated area in ha) × 15 plots per ha
                luas_rk_sum    = intersecting['LUAS_RK'].sum()
                target_n_plots = int(luas_rk_sum * 15)
                print(f"  LUAS_RK total: {luas_rk_sum:.2f} ha → target {target_n_plots} plots")
            except Exception as e:
                print(f"  ⚠️  Could not read LUAS_RK ({e}) — defaulting to 150 plots")
                target_n_plots = 150

            # Create 100 m sampling grid, clip to detected Acrostichum area
            grid_100m      = self.create_qgis_style_grid(
                clipped_acrostichum.total_bounds, 100, 100, crs=georeference['crs']
            )
            clipped_grid   = gpd.clip(grid_100m, clipped_acrostichum)
            random_polygons_gdf = self.place_polygons_in_clipped_grids(
                clipped_grid, clipped_acrostichum,
                polygon_cell_size, spacing_m=2.0,
                target_total_polygons=target_n_plots
            )
            self.save_to_shapefile(random_polygons_gdf, "Random_Polygons_Within_Acrostichum.shp")

        # Remove temporary tile files to keep the working directory clean
        print("\n🧹 Removing temporary tile files...")
        for tile in tiles:
            if os.path.exists(tile['filename']):
                os.remove(tile['filename'])

        self.create_overview_visualization_with_random(
            image_path, clipped_acrostichum, mangrove_gdf,
            random_polygons_gdf, georeference, pixel_bounds
        )

        print("\n✅ Delineation pipeline complete.")
        print("=" * 70)


print("✓ VegetationDelineator class defined and ready")

## Step 3: Configure and Run the Pipeline

Update the **USER CONFIGURATION** block below with your own file paths and Roboflow API key, then run the cell.

> **Security note:** Never commit your API key to a public Git repository. Consider loading it from an environment variable:
> ```python
> import os
> ROBOFLOW_API_KEY = os.environ["ROBOFLOW_API_KEY"]
> ```

### Key parameters to tune

| Parameter | Default | Effect |
|---|---|---|
| `confidence` | `0.4` | Lower = more detections (more false positives); higher = fewer (more false negatives) |
| `merge_distance` | `50` | Pixel radius for DBSCAN; increase if patches are large and fragmented |
| `polygon_cell_size` | `(5, 10)` | Dimensions (m) of each random sample plot |

In [ ]:
# =====================================================
# USER CONFIGURATION — update these before running
# =====================================================

# Your Roboflow account API key (find at app.roboflow.com → Settings → API Keys)
# ⚠️  Do NOT share or commit this key publicly
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"   # <-- CHANGE THIS

# Full path to the drone orthomosaic GeoTIFF
IMAGE_PATH = r"D:\M4CR\Desa Selatnama\KTH Parpi Jaya & KTH Makmur Jaya_Selatnama.tif"

# Full path to the study-area boundary Shapefile
SHAPEFILE_PATH = r"C:\Users\ACER\Downloads\PRM_RIAU_2025_1064 (1)\Selatnama.shp"

# =====================================================

# --- Pre-flight checks: verify inputs before starting the long pipeline ---
errors = []
if not os.path.exists(IMAGE_PATH):
    errors.append(f"Image not found: {IMAGE_PATH}")
if not os.path.exists(SHAPEFILE_PATH):
    errors.append(f"Shapefile not found: {SHAPEFILE_PATH}")
if ROBOFLOW_API_KEY == "YOUR_ROBOFLOW_API_KEY" or not ROBOFLOW_API_KEY:
    errors.append("ROBOFLOW_API_KEY has not been set — please update it above")

if errors:
    for err in errors:
        print(f"❌ {err}")
else:
    print(f"Image     : {IMAGE_PATH}")
    print(f"Shapefile : {SHAPEFILE_PATH}")

    try:
        # Initialise the delineator — downloads the Roboflow model weights
        delineator = VegetationDelineator(
            api_key    = ROBOFLOW_API_KEY,
            project_id = "acrostichum-2-g8ewp",
            version    = 6
        )

        # Run the complete pipeline
        delineator.run_full_delineation(
            image_path         = IMAGE_PATH,
            shapefile_path     = SHAPEFILE_PATH,
            confidence         = 0.4,     # Detection confidence threshold (0–1)
            merge_distance     = 50,      # DBSCAN merge radius in pixels
            create_random_grid = True,    # Generate random 5×10 m sample plots
            polygon_cell_size  = (5, 10)  # Width × Height of each sample plot (metres)
        )

    except Exception as e:
        import traceback
        print(f"\n❌ Pipeline error: {e}")
        traceback.print_exc()